In [12]:
import pandas as pd

df_test = pd.read_csv("test_data.csv")

df_test.head()

,datapointID,textA,textB
0,120,o utse airs eaien eii llin cri.,NaN
1,1168,e str er's ga ii hie ied al wi e aad en swih e...,NaN
2,796,"den diin eipri skis, eie ren ers ie taend ie c...",NaN
3,114,he e kekcleazologuclea nem e hejen eleajecleaz...,NaN
4,989,licrhcrevbe coheleaenlicr gleaoup crhiileaed i...,NaN


In [13]:
import re

df_test.textB = df_test.textB.fillna("empty")
def preprocess(text):
    if text == 'empty':
        return text
    text = re.sub(r'[^\w\s]', "", text)
    text = re.sub(r'\d+', '', text)
    return text.lower().strip()

df_test.textA = df_test.textA.apply(preprocess)
df_test.textB = df_test.textB.apply(preprocess)

df_test.head()

,datapointID,textA,textB
0,120,o utse airs eaien eii llin cri,empty
1,1168,e str ers ga ii hie ied al wi e aad en swih es...,empty
2,796,den diin eipri skis eie ren ers ie taend ie ca...,empty
3,114,he e kekcleazologuclea nem e hejen eleajecleaz...,empty
4,989,licrhcrevbe coheleaenlicr gleaoup crhiileaed i...,empty


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

A = df_test["textA"]
B = df_test[df_test.textB != "empty"].textB

vect = TfidfVectorizer()
A = vect.fit_transform(A)
B = vect.transform(B)

A

<1587x11097 sparse matrix of type '<class 'numpy.float64'>'
	with 26125 stored elements in Compressed Sparse Row format>

In [15]:
from sklearn.cluster import KMeans

kmeans = KMeans(4, random_state=42)

kmeans.fit(A)
df_test["clusterA"] = kmeans.predict(A)

preds = kmeans.predict(B)

In [16]:
ids = df_test[df_test.textB != "empty"].datapointID

df_B = pd.DataFrame({
    "ids": ids,
    "preds": preds
})
df_B.head(n=2)

,ids,preds
7,1553,0
8,1537,0


In [17]:
for ID in df_test.datapointID:
    row_mask = df_test["datapointID"] == ID
    rowB = df_B.loc[df_B["ids"] == ID]

    if df_test.loc[row_mask, "textB"].values[0] == "empty":
        df_test.loc[row_mask, "clusterB"] = -1
    else:
        pred_value = rowB["preds"].values[0]
        df_test.loc[row_mask, "clusterB"] = int(pred_value)

df_test.head(n=10)

,datapointID,textA,textB,clusterA,clusterB
0,120,o utse airs eaien eii llin cri,empty,3,-1.0
1,1168,e str ers ga ii hie ied al wi e aad en swih es...,empty,3,-1.0
2,796,den diin eipri skis eie ren ers ie taend ie ca...,empty,3,-1.0
3,114,he e kekcleazologuclea nem e hejen eleajecleaz...,empty,2,-1.0
4,989,licrhcrevbe coheleaenlicr gleaoup crhiileaed i...,empty,0,-1.0
5,614,celeaemeleaguclea egiu enjoukoleabe oklevelbol...,empty,1,-1.0
6,1057,iidvenlicr of gunpodelea \n\niilealicrbellelea...,empty,0,-1.0
7,1553,unec tler,evenlicrcr cruch iicr feiicrlicrcr biinquelicr...,0,0.0
8,1537,giri seil enc heil,folealicr longueubel bubellicr fleaom licro ...,0,0.0
9,577,iicr ii leaecrullicr licrleaue ciicrlicrlecr e...,empty,0,-1.0


In [20]:
answers2 = []
ids = []

for i in range (1, 1238):
    row_mask = df_test["datapointID"] == i
    ids.append(i)
    answers2.append(df_test.loc[row_mask, "clusterA"].values[0])

len(answers2)

1237

In [18]:
sample = pd.read_csv("sample_output.csv")

ids = []
answers = []

for ID in sample.loc[sample.subtaskID == 1].datapointID:
    row_mask = df_test["datapointID"] == ID
    clusA = df_test.loc[row_mask, "clusterA"].values[0]
    clusB = df_test.loc[row_mask, "clusterB"].values[0]
    ids.append(ID)
    if clusA == clusB:
        answers.append(True)
    else:
        answers.append(False)

subtask1 = pd.DataFrame({
    "datapointID": ids,
    "subtaskID": 1,
    "answer": answers
})

subtask1.head()

,datapointID,subtaskID,answer
0,1238,1,False
1,1239,1,False
2,1240,1,True
3,1241,1,True
4,1242,1,False


In [21]:
subtask2 = pd.DataFrame({
    "datapointID": ids,
    "subtaskID": 2,
    "answer": answers2
})

subtask2.answer = subtask2.answer.map({0: "Englcrevbeh", 1: "Hungeleabeen", 2: "En Gli", 3: "Hure"})

submission = pd.concat([subtask1, subtask2])
submission.to_csv("output.csv", index=False)
submission

,datapointID,subtaskID,answer
0,1238,1,False
1,1239,1,False
2,1240,1,True
3,1241,1,True
4,1242,1,False
...,...,...,...
1232,1233,2,Englcrevbeh
1233,1234,2,Englcrevbeh
1234,1235,2,Hure
1235,1236,2,Hure


In [22]:
subtask2.answer.value_counts()

answer
Englcrevbeh     574
Hungeleabeen    288
Hure            280
En Gli           95
Name: count, dtype: int64